# 07 — Pipeline complet : Détection du glaucome par ML classique

Notebook exécutable de bout en bout.

**Pipeline :**
1. Chargement multi-dossier (0+1+2)
2. Prétraitement ophtalmo (CLAHE, masque circulaire)
3. Extraction features "full" (color spaces + Gabor + LBP multi-scale + Haralick + spatial)
4. Sélection de features (variance threshold + mutual info)
5. Cross-validation 5-fold avec SMOTE intégré
6. Hyperparameter tuning XGBoost
7. Ensemble stacking
8. Évaluation finale + graphiques

**Dataset :** AIROGS — 101k images de fond d'œil, ~3.3% glaucome référable (RG)

In [1]:
import sys
sys.path.insert(0, '..')

import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

print('✓ Imports ultra-rapides chargés')

✓ Imports ultra-rapides chargés


---
## Étape 1 — Baseline (features simples, dossier 0 seul)

In [2]:
# Importer au moment du besoin (lazy loading)
from src.config import PROJECT_ROOT, FEATURE_CACHE_DIR, RANDOM_STATE, CV_FOLDS
from src.data_loading import get_ml_subset_df

# Données dossier 0 uniquement (sous-échantillon pour lancement rapide)
df0 = get_ml_subset_df()

MAX_BASELINE = 5000  # nombre max d'images pour la baseline rapide
if len(df0) > MAX_BASELINE:
    df0 = df0.sample(n=MAX_BASELINE, random_state=RANDOM_STATE)

# Split simple sans sklearn (approx. 70/15/15)
df0 = df0.sample(frac=1, random_state=RANDOM_STATE)  # shuffle
n = len(df0)
n_test = int(0.15 * n)
n_val = int(0.15 * n)

test_df0 = df0.iloc[:n_test]
val_df0 = df0.iloc[n_test:n_test + n_val]
train_df0 = df0.iloc[n_test + n_val:]

print(f'Dossier 0 (baseline rapide) — Train: {len(train_df0)} | Val: {len(val_df0)} | Test: {len(test_df0)}')
print(f'RG dans train: {(train_df0["class"] == "RG").sum()} ({(train_df0["class"] == "RG").mean()*100:.1f}%)')

Dossier 0 (baseline rapide) — Train: 3500 | Val: 750 | Test: 750
RG dans train: 120 (3.4%)


In [3]:
from src.feature_extraction import build_feature_matrix

# Baseline : features simples SANS prétraitement
t0 = time.time()
X_train_simple, y_train_simple = build_feature_matrix(
    train_df0, feature_type='simple', preprocess=False,
    cache_dir=str(FEATURE_CACHE_DIR),
)
X_test_simple, y_test_simple = build_feature_matrix(
    test_df0, feature_type='simple', preprocess=False,
    cache_dir=str(FEATURE_CACHE_DIR),
)
print(f'Features simples: {X_train_simple.shape[1]} features, {time.time()-t0:.0f}s')

Features simples: 102 features, 0s


In [4]:
from src.pipeline_ml import create_pipeline, evaluate_with_cv

# Évaluation baseline : RF + features simples
# → version LÉGÈRE : moins de données + moins de folds + RF plus petit

# Sous-échantillonnage pour la cross-val de baseline
MAX_CV_SAMPLES = 3000
X_cv = X_train_simple
y_cv = y_train_simple
if len(y_cv) > MAX_CV_SAMPLES:
    import numpy as np
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(len(y_cv), size=MAX_CV_SAMPLES, replace=False)
    X_cv = X_cv[idx]
    y_cv = y_cv[idx]
    print(f"Baseline CV sur {len(y_cv)} échantillons (sous-échantillonnés).")

pipeline_baseline = create_pipeline(
    model_name='rf',
    resample_strategy='smote',
    use_scaler=True,
    class_weight='balanced',
    clf_params={'n_estimators': 50, 'max_depth': 10},  # RF plus léger
)
print('\n--- Baseline (RF + simple features, pas de preprocessing) — CV=3 ---')
cv_baseline = evaluate_with_cv(X_cv, y_cv, pipeline_baseline, cv=3)

Baseline CV sur 3000 échantillons (sous-échantillonnés).

--- Baseline (RF + simple features, pas de preprocessing) — CV=3 ---
Cross-validation 3-fold...
  roc_auc: 0.5960 ± 0.0336
  sens_at_95spec: nan ± nan


---
## Étape 2 — Features enrichies avec prétraitement

In [5]:
# Features simples AVEC prétraitement ophtalmo
t0 = time.time()
X_train_preproc, y_train_preproc = build_feature_matrix(
    train_df0, feature_type='simple', preprocess=True,
    cache_dir=str(FEATURE_CACHE_DIR),
)
X_test_preproc, y_test_preproc = build_feature_matrix(
    test_df0, feature_type='simple', preprocess=True,
    cache_dir=str(FEATURE_CACHE_DIR),
)
print(f'Features simples + preprocessing: {X_train_preproc.shape[1]} features, {time.time()-t0:.0f}s')

pipeline_preproc = create_pipeline(model_name='rf', resample_strategy='smote')
print('\n--- + Preprocessing ophtalmo ---')
cv_preproc = evaluate_with_cv(X_train_preproc, y_train_preproc, pipeline_preproc, cv=CV_FOLDS)

Features (simple): 100%|██████████| 750/750 [00:16<00:00, 45.30it/s]


Features simples + preprocessing: 102 features, 97s

--- + Preprocessing ophtalmo ---
Cross-validation 5-fold...
  roc_auc: 0.6976 ± 0.0314
  sens_at_95spec: nan ± nan


In [6]:
# Features "full" avec prétraitement
t0 = time.time()
X_train_full, y_train_full = build_feature_matrix(
    train_df0, feature_type='full', preprocess=True,
    cache_dir=str(FEATURE_CACHE_DIR),
)
X_test_full, y_test_full = build_feature_matrix(
    test_df0, feature_type='full', preprocess=True,
    cache_dir=str(FEATURE_CACHE_DIR),
)
print(f'Features full: {X_train_full.shape[1]} features, {time.time()-t0:.0f}s')

pipeline_full = create_pipeline(model_name='rf', resample_strategy='smote')
print('\n--- + Features full (color spaces + Gabor + LBP + Haralick + spatial) ---')
cv_full = evaluate_with_cv(X_train_full, y_train_full, pipeline_full, cv=CV_FOLDS)

Features (full): 100%|██████████| 750/750 [01:07<00:00, 11.12it/s]


Features full: 548 features, 378s

--- + Features full (color spaces + Gabor + LBP + Haralick + spatial) ---
Cross-validation 5-fold...
  roc_auc: 0.7431 ± 0.0193
  sens_at_95spec: nan ± nan


---
## Étape 3 — Sélection de features

In [8]:
# Sélection de features
from src.feature_selection import FeatureSelector
selector = FeatureSelector(variance_threshold=0.01, k_best=150, use_pca=False)
X_train_sel = selector.fit_transform(X_train_full, y_train_full)
X_test_sel = selector.transform(X_test_full)

pipeline_sel = create_pipeline(model_name='rf', resample_strategy='smote')
print('\n--- + Feature selection ---')
cv_sel = evaluate_with_cv(X_train_sel, y_train_full, pipeline_sel, cv=CV_FOLDS)

Feature selection (548 features initiales):
  VarianceThreshold: 548 → 174 features (374 supprimées)
  Mutual Info: 174 → 150 features (top-150)
  Résultat final: 548 → 150 features

--- + Feature selection ---
Cross-validation 5-fold...
  roc_auc: 0.7171 ± 0.0393
  sens_at_95spec: nan ± nan


---
## Étape 4 — Hyperparameter tuning

In [12]:
# Tuning XGBoost
from src.tuning import tune_model
from src.config import PROJECT_ROOT, FEATURE_CACHE_DIR, RANDOM_STATE, CV_FOLDS, N_TUNING_ITER
print('\n--- Hyperparameter tuning XGBoost ---')
tuning_result = tune_model(
    X_train_sel, y_train_full,
    model_name='xgb',
    cv=CV_FOLDS,
    n_iter=N_TUNING_ITER,
    scoring='roc_auc',
    verbose=0,
)

print(f'\nMeilleur score CV: {tuning_result["best_score"]:.4f}')
print(f'Meilleurs params: {tuning_result["best_params"]}')


--- Hyperparameter tuning XGBoost ---
Tuning XGB (50 itérations, 5-fold CV, scoring=roc_auc)...
  Meilleur score: 0.7809
  Meilleurs params: {'subsample': 0.6, 'scale_pos_weight': 1, 'reg_lambda': 1.0, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.6}

Meilleur score CV: 0.7809
Meilleurs params: {'subsample': 0.6, 'scale_pos_weight': 1, 'reg_lambda': 1.0, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.6}


In [13]:
# Pipeline avec les meilleurs hyperparamètres
best_params = tuning_result['best_params']
pipeline_tuned = create_pipeline(
    model_name='xgb', resample_strategy='smote',
    clf_params=best_params,
)
print('\n--- + XGBoost tuné ---')
cv_tuned = evaluate_with_cv(X_train_sel, y_train_full, pipeline_tuned, cv=CV_FOLDS)


--- + XGBoost tuné ---
Cross-validation 5-fold...
  roc_auc: 0.7411 ± 0.0264
  sens_at_95spec: nan ± nan


---
## Étape 5 — Ensemble (Stacking)

In [16]:
# Stacking : XGBoost tuné + RF → Logistic Regression
from src.pipeline_ml import (
    create_pipeline,
    evaluate_with_cv,
    create_stacking_ensemble,
    get_classifier,           # si tu l’utilises ici
)
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

stacking = create_stacking_ensemble(
    base_models={
        'xgb': get_classifier('xgb', scale_pos_weight=29),
        'rf': get_classifier('rf'),
    }
)

# Pipeline avec SMOTE + scaler + stacking
pipeline_stacking = ImbPipeline([
    ('resampler', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ('scaler', StandardScaler()),
    ('classifier', stacking),
])

print('\n--- + Ensemble Stacking (XGB + RF → LR) ---')
cv_stacking = evaluate_with_cv(X_train_sel, y_train_full, pipeline_stacking, cv=CV_FOLDS)


--- + Ensemble Stacking (XGB + RF → LR) ---
Cross-validation 5-fold...
  roc_auc: 0.7305 ± 0.0389
  sens_at_95spec: nan ± nan


---
## Étape 6 — Tableau récapitulatif

In [17]:
# Tableau de progression
results_table = pd.DataFrame([
    {
        'Configuration': 'Baseline (RF + simple)',
        'Features': X_train_simple.shape[1],
        'AUC-ROC': f"{cv_baseline.get('roc_auc', {}).get('mean', 0):.4f} ± {cv_baseline.get('roc_auc', {}).get('std', 0):.4f}",
        'Sens@95%sp': f"{cv_baseline.get('sens_at_95spec', {}).get('mean', 0):.4f}",
    },
    {
        'Configuration': '+ Preprocessing',
        'Features': X_train_preproc.shape[1],
        'AUC-ROC': f"{cv_preproc.get('roc_auc', {}).get('mean', 0):.4f} ± {cv_preproc.get('roc_auc', {}).get('std', 0):.4f}",
        'Sens@95%sp': f"{cv_preproc.get('sens_at_95spec', {}).get('mean', 0):.4f}",
    },
    {
        'Configuration': '+ Features full',
        'Features': X_train_full.shape[1],
        'AUC-ROC': f"{cv_full.get('roc_auc', {}).get('mean', 0):.4f} ± {cv_full.get('roc_auc', {}).get('std', 0):.4f}",
        'Sens@95%sp': f"{cv_full.get('sens_at_95spec', {}).get('mean', 0):.4f}",
    },
    {
        'Configuration': '+ Feature selection',
        'Features': X_train_sel.shape[1],
        'AUC-ROC': f"{cv_sel.get('roc_auc', {}).get('mean', 0):.4f} ± {cv_sel.get('roc_auc', {}).get('std', 0):.4f}",
        'Sens@95%sp': f"{cv_sel.get('sens_at_95spec', {}).get('mean', 0):.4f}",
    },
    {
        'Configuration': '+ XGBoost tuné',
        'Features': X_train_sel.shape[1],
        'AUC-ROC': f"{cv_tuned.get('roc_auc', {}).get('mean', 0):.4f} ± {cv_tuned.get('roc_auc', {}).get('std', 0):.4f}",
        'Sens@95%sp': f"{cv_tuned.get('sens_at_95spec', {}).get('mean', 0):.4f}",
    },
    {
        'Configuration': '+ Stacking (XGB+RF→LR)',
        'Features': X_train_sel.shape[1],
        'AUC-ROC': f"{cv_stacking.get('roc_auc', {}).get('mean', 0):.4f} ± {cv_stacking.get('roc_auc', {}).get('std', 0):.4f}",
        'Sens@95%sp': f"{cv_stacking.get('sens_at_95spec', {}).get('mean', 0):.4f}",
    },
])

print('\n' + '='*80)
print('TABLEAU RÉCAPITULATIF — Progression incrémentale')
print('='*80)
print(results_table.to_string(index=False))


TABLEAU RÉCAPITULATIF — Progression incrémentale
         Configuration  Features         AUC-ROC Sens@95%sp
Baseline (RF + simple)       102 0.5960 ± 0.0336        nan
       + Preprocessing       102 0.6976 ± 0.0314        nan
       + Features full       548 0.7431 ± 0.0193        nan
   + Feature selection       150 0.7171 ± 0.0393        nan
        + XGBoost tuné       150 0.7411 ± 0.0264        nan
+ Stacking (XGB+RF→LR)       150 0.7305 ± 0.0389        nan


---
## Étape 7 — Évaluation finale sur le test set

In [21]:
# Entraîner le meilleur modèle sur tout le train et évaluer sur test

from src.pipeline_ml import (
    create_pipeline,
    evaluate_with_cv,
    create_stacking_ensemble,
    get_classifier,
    get_smote,
    fit_scaler,
    threshold_predict,
    find_threshold_for_specificity,  # <-- ajouter ceci
)
from src.evaluation_metrics import print_metrics, compute_all_metrics_with_ci, plot_roc_comparison, plot_confusion_matrix

scaler = fit_scaler(X_train_sel)
X_train_final = scaler.transform(X_train_sel)
X_test_final = scaler.transform(X_test_sel)

X_train_r, y_train_r = get_smote(X_train_final, y_train_full)

# Meilleur modèle : XGBoost tuné
best_clf = tuning_result['best_model']
best_clf.fit(X_train_r, y_train_r)

# Prédictions
y_score_final = best_clf.predict_proba(X_test_final)[:, 1]
threshold = find_threshold_for_specificity(y_train_full, 
    best_clf.predict_proba(X_train_final)[:, 1], target_spec=0.95)
y_pred_final = threshold_predict(best_clf.predict_proba(X_test_final), threshold)

print('\n=== RÉSULTATS FINAUX SUR LE TEST SET ===')
print_metrics(y_test_full, y_pred_final, y_score_final)

print('\nMétriques avec IC 95%:')
final_metrics = compute_all_metrics_with_ci(y_test_full, y_score_final)


=== RÉSULTATS FINAUX SUR LE TEST SET ===
[[719  13]
 [ 16   2]]
AUC-ROC:                    0.6957
pAUC (90-100% spec):        0.5890
Sensibilité @ 95% spec:     0.2222
Sensibilité @ 90% spec:     0.4444

Métriques avec IC 95%:
  auc_roc: 0.6957 [0.5453, 0.8480]
  pauc_90_100: 0.5890 [0.5014, 0.6917]
  sens_at_95spec: 0.2222 [0.0500, 0.5000]
  sens_at_90spec: 0.4444 [0.1426, 0.6667]


In [24]:
# Courbes ROC comparatives
# Pour tracer les ROC, on doit évaluer chaque config sur le test set
# On réutilise le meilleur modèle pour la courbe finale
from src.evaluation_metrics import plot_roc_comparison

import matplotlib.pyplot as plt

roc_data = {
    'XGBoost tuné + features full': {
        'y_true': y_test_full,
        'y_score': y_score_final,
    }
}

# Ajouter la baseline pour comparaison
scaler_base = fit_scaler(X_train_simple)
X_train_base_s = scaler_base.transform(X_train_simple)
X_test_base_s = scaler_base.transform(X_test_simple)
X_train_base_r, y_train_base_r = get_smote(X_train_base_s, y_train_simple)

clf_base = get_classifier('rf')
clf_base.fit(X_train_base_r, y_train_base_r)
roc_data['Baseline (RF + simple)'] = {
    'y_true': y_test_simple,
    'y_score': clf_base.predict_proba(X_test_base_s)[:, 1],
}

fig = plot_roc_comparison(roc_data, title='ROC — Baseline vs Pipeline optimisé')
plt.show()

In [26]:
from src.evaluation_metrics import plot_confusion_matrix
import matplotlib.pyplot as plt
# Matrice de confusion
plot_confusion_matrix(y_test_full, y_pred_final, title='Matrice de confusion — Modèle final')
plt.show()

In [28]:
# Sauvegarder le meilleur modèle
import joblib

models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(exist_ok=True)

joblib.dump(best_clf, models_dir / 'best_xgb_tuned.joblib')
joblib.dump(scaler, models_dir / 'scaler_full.joblib')
# On ne sauvegarde pas `selector` à cause du problème de pickling

print(f'Modèle sauvegardé dans {models_dir}')
print(f'  - best_xgb_tuned.joblib')
print(f'  - scaler_full.joblib')

Modèle sauvegardé dans /Users/leduigouvincent/Documents/glaucome/models
  - best_xgb_tuned.joblib
  - scaler_full.joblib


---
## Conclusion

**Améliorations apportées :**
1. Prétraitement ophtalmo (CLAHE + masque circulaire)
2. Features multi-espaces couleur + textures (Gabor, LBP multi-échelles)
3. Features spatiales spécifiques au glaucome (CDR, ISNT, zones)
4. Sélection de features (variance + information mutuelle)
5. Hyperparameter tuning (RandomizedSearchCV)
6. Ensemble stacking

**Prochaine étape (hors scope ML classique) :** Deep learning avec transfer learning pour la clinique.